In [1]:
import lightgbm as lgb
import numpy as np
from scipy.stats import norm
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
train = pd.read_parquet("../data/model_input/train.parquet")
val = pd.read_parquet("../data/model_input/validation.parquet")
test = pd.read_parquet("../data/model_input/test.parquet")

In [3]:
FEATURE_COLS = [
    'net_gex_norm', 'net_dex_norm', 'net_tex_norm',
    'atm_iv', 'iv_call_25d', 'iv_put_25d', 'iv_skew_25d', 'iv_smile_curvature_25d',
    'ttm_min', 'theta_decay', 'log_return_from_open',
    'oi_concentration_top3', 'put_oi_fraction', 'atm_spread_norm',
    'distance_to_max_oi_norm', 'put_max_oi_strike_norm', 'call_max_oi_strike_norm',
]
TARGET = 'log_return_norm'
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

In [4]:
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
params = dict(objective='quantile', learning_rate=0.03, num_leaves=15,
              min_child_samples=100, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, n_estimators=2000, verbose=-1)

models = {}
for q in QUANTILES:
    models[q] = lgb.LGBMRegressor(alpha=q, **params).fit(
        X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='quantile',
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

Z = np.column_stack([models[q].predict(X_test) for q in QUANTILES])
Z = np.sort(Z, axis=1)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's quantile: 0.20973
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's quantile: 0.395351
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's quantile: 0.462773
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's quantile: 0.37355
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's quantile: 0.208097
